## Project Objectives
1. Build an efficient SQL database to analyze the Global Financial Crisis using Barra data
2. Demonstrate mastery of SQL data warehousing concepts:
   - Proper schema design with multiple tables
   - Data type optimization and NULL handling
   - Primary keys and composite indices
   - Complex SQL queries using CTEs, Views, and Subqueries
   - JOIN operations (INNER, LEFT, and simulated FULL OUTER)
   - Transaction management with ROLLBACK/COMMIT

## Important Data Considerations

### Column Naming Best Practices
The Barra dataset includes columns with '%' in their names (e.g., 'SRISK%', 'TRISK%', 'WGT1%'). Consider:
- Is this best practice for database column naming?
- How should you handle these in your schema?
- Document your approach and any renaming decisions

### Estimation Universe Filter
- Use stocks in the 2007 estimation universe: `E3ESTU = 1` or `NONESTU = 0`
- These represent investible stocks in 2007
- When calculating percent changes to 2008, include all stocks to avoid survivorship bias
- Document how you handle companies that become uninvestible

### 2. Risk Metrics Analysis (25%)
**Key Decisions Required:**

**Beta Comparison:**
- Compare 'HBTA' vs 'BETA' columns
- Research differences in the Barra handbook
- Recommend which the risk management division should use with justification

**Risk Measure Comparison:**
- Compare 'SRISK%' vs 'TRISK%'
- Research differences in the Barra handbook
- Recommend which measure for risk assessment with justification

**Risk Prediction Model:**
Using SQL, calculate mean and standard deviation for potential risk indicators:
- TRADEACT (trading activity)
- VOLTILTY (volatility)
- MOMENTUM
- VALUE
- EARNVAR (earnings variability)
- LEVERAGE

Determine which factors (individually or in combination) best predict PCT_CHANGE between 2007-2008.

### 3. Transaction Management (15%)
**Scenario**: Your risk management department has a "crystal ball" predicting the 2008 crisis.

Demonstrate a transaction that:
* Identifies Bear Stearns and Lehman Brothers BARRIDs
* Temporarily adjusts their risk metrics based on "prediction"
* Analyzes impact on portfolio risk
* Uses ROLLBACK to restore original data
* Documents the what-if analysis results

### **4. Sector Analysis (20%)**
Financial Services Deep Dive:
- Use your Industry-to-Sector mapping table to query which industries belong to the Financial Services sector
- Use the resulting industry list in your WHERE clause with INDNAME1
- Analyze sector-specific risk patterns
- Compare Financial Services sector performance against other sectors

### 5. Complex Query Techniques (20%)
Implement the same analysis using all three methods:
* **CTE (WITH clause)**: For multi-step calculations
* **VIEW**: For reusable logic
* **SUBQUERY**: For inline calculations

Compare performance and readability, recommend the best approach.

Example of query to identify risky companies to a hypothetical risk committee:
- Calculate z-scores for multiple risk factors
- Identify outliers within each sector (by cross referencing INDNAME1 with the Industry-to-Sector table).
- Rank companies by composite risk score

### 6. JOIN Operations Showcase (20%)
Demonstrate these JOIN types with meaningful analysis:
* **INNER JOIN**: Companies surviving both years
* **LEFT JOIN**: All 2007 companies, showing which disappeared
* **Simulated FULL OUTER JOIN**: Complete picture using UNION approach
* Show proper NULL handling in JOIN results

## Analysis Requirements

### Core Analyses (Complete ALL)

1. **Risk Factor Effectiveness**
   - Calculate statistics directly in SQL (mean, stddev)
   - Test each risk factor's predictive power
   - Identify optimal combination of factors
   - Use window functions for percentile rankings

2. **Survivor Analysis**
   - Track companies from 2007 to 2008
   - Avoid survivorship bias in calculations
   - Identify characteristics of survivors vs failures

3. **Sector Impact Study**
  - Complete the industry-to-sector mapping table from the Barra handbook
  - Query your mapping table to identify all industries in each sector
  - Use SQL JOINs between your mapping table and Barra data tables
  - Example query structure:
    - First: `SELECT INDNAME FROM ind_to_sectors WHERE SECTOR = 'Financials'`
    - Then use result in: `WHERE INDNAME1 IN (subquery or join result)`

4. **Beta and Risk Comparison**
   - Statistical comparison of HBTA vs BETA
   - Statistical comparison of SRISK% vs TRISK%
   - Recommendation with evidence

## Deliverables

### 1. Code Notebook
* Schema creation with column naming decisions
* Index creation and EXPLAIN plan analysis
* Transaction examples with Bear Stearns/Lehman scenarios
* All three query methods comparison
* Risk factor statistical calculations in SQL
* JOIN demonstrations with NULL handling

### 2. Database Documentation
* ERD showing all tables and relationships
* Column naming convention and any modifications
* Data type justifications
* Handling of estimation universe filtering
* NULL handling strategy

### 3. Analysis Report
* Executive summary of risk factor findings
* Beta and risk measure recommendations
* Sector impact analysis
* Visualizations (using Python after SQL retrieval)
* Performance metrics for different query approaches

### 4. Presentation Video (10-15 minutes)
* Live demonstration of:
  - Transaction with Bear Stearns/Lehman scenario
  - Risk factor statistical calculations
  - JOIN operations showing survivor analysis
  - Query method comparison (CTE vs View vs Subquery)
* Key insights about predictive risk factors
* Technical decisions and trade-offs

### 5. Trello Board
* Task assignments and progress
* Query development tracking
* Testing and optimization notes

## Grading Rubric

| Component | Weight | Details |
|-----------|--------|---------|
| Schema Design | 20% | Column naming, data types, keys, indices |
| Risk Analysis | 25% | Beta/risk comparison, factor effectiveness |
| SQL Techniques | 20% | CTEs, Views, Subqueries, JOINs |
| Transaction Management | 15% | Meaningful what-if scenarios |
| Sector Analysis | 15% | Complete mapping, financial services focus |
| Documentation | 5% | Clear, complete, professional |

## Key Requirements and Hints
* Build the complete industry-to-sector mapping table from Barra handbook PDF pages 83-84
  * Use SQL JOINs with your mapping table rather than hard-coding industry lists
  * All sector analysis should query the mapping table first, then join with Barra data
  * This demonstrates proper database normalization and JOIN usage
* Calculate all statistics (mean, stddev) directly in SQL, not Python
* Handle estimation universe filtering properly to avoid bias
* Performance matters - use EXPLAIN to show optimization
* Your risk predictions should be data-driven and justified
* Focus on SQL mastery, not financial theory

## Bonus Opportunities (up to 10% extra)
* Implement composite risk score using multiple factors
* Create automated column renaming system for best practices
* Build cross-validation framework in SQL
* Develop early warning system based on your findings
* Create interactive risk dashboard with parameterized queries

Remember: This project tests your SQL and data warehousing skills. Use the financial crisis as a compelling context to demonstrate sophisticated database techniques and analytical thinking.

In [ ]:
# Because jupyter notebook has a global name space; use this area for imports
import pandas as pd
import numpy as np
import plotly.express as px
import math
from typing import Union

## SQLAlchemy imports
from sqlalchemy import create_engine
from sqlalchemy.types import Date, Float, String, Integer, Boolean

# aux packages
from IPython.display import Image

# Obtain resource files

In [ ]:
import pathlib
import importlib.util

#check if gdown is installed in current python environment; alert the user to activate their virtual environment to avoid installing to global
if importlib.util.find_spec("gdown") is None:
    print("gdown is not installed. Please activate your virtual environment and install gdown to proceed.")

# dictionary: filename and gdown id
file_names = {'USE3L0712.RSK': '1Y0WE4cqbZfdNEvFuWIDR_d6oDbHHNkly',
              'USE3L0812.RSK': '1Z3vdd5m8uoB4whomq92DQbZHJiKYJc0v'}

curr_working_dir = pathlib.Path.cwd()
print(f"Current working directory: {curr_working_dir}")

for file_name, gdown_id in file_names.items():
    file_path = curr_working_dir / file_name
    if not file_path.exists():
        print(f"File {file_path} does not exist.")
        !gdown {gdown_id} -O {file_path}
    else:
        print(f"File {file_path} already exists. No need to download.")




### 1. Database Schema Design (20%)
* Create at least 3 tables:
  - 2007 Barra USE3 data
  - 2008 Barra USE3 data  
  - Industry-to-Sector mapping (from PDF pages 83-84)
* Use the Industry-to-Sector table for all sector-related queries
* Address column naming issues (columns with '%')
* Optimize data types with justification
* Implement primary keys and foreign key relationships
* Create both single and composite indices

In [ ]:
# check file integrity of downloaded files
for file_count, file_name in enumerate(file_names.keys(), start=1):
    print(f"\nChecking integrity of file {file_count}: {file_name}...")
    ! head -n 3 {file_name}